---
#### Import libraries and load data

* Loaded the cleaned and feature-selected dataset prepared during the preprocessing phase.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

In [2]:
data = pd.read_csv("data_for_modeling.csv")
data.head()

,points_share,VORP,PER,FG,2P,DWS,USG%,WS/48,WS,PTS,FTA
0,0.000,1.7,15.9,5.8,3.0,1.8,21.7,0.150,6.6,17.1,3.1
1,0.000,1.3,20.9,3.0,2.9,1.7,21.7,0.168,3.5,6.8,1.1
2,0.000,0.0,2.9,0.3,0.3,0.0,20.0,-0.083,0.0,1.3,0.7
3,0.056,5.5,24.4,6.7,5.2,2.9,24.2,0.211,10.7,18.5,4.2
4,0.000,4.2,20.6,7.4,4.5,2.0,27.4,0.150,8.6,22.1,5.3


---
#### Analyze dataset

* The dataset includes 12,697 player-season rows and 11 features, covering both traditional and advanced statistics.
* The target variable points_share has a mean of 0.018 and a max of 1.0, indicating that most players did not receive All-NBA votes, while a few received a full vote share.
* Stats like PTS (points per game), USG% (usage rate), and WS (win shares) show wide variance, reflecting a mix of role players and stars.
* PER (Player Efficiency Rating) and VORP (Value Over Replacement Player) span large ranges, with outliers suggesting extremely poor or elite seasons.
* Some minimum values are negative (e.g., VORP, DWS, WS), which is expected in advanced metrics and may influence model interpretation.

In [49]:
data.shape

(12697, 11)

In [3]:
data.describe()

,points_share,VORP,PER,FG,2P,DWS,USG%,WS/48,WS,PTS,FTA
count,12697.000000,12697.000000,12697.000000,12697.000000,12697.000000,12697.000000,12697.000000,12697.000000,12697.000000,12697.000000,12697.000000
mean,0.017643,0.581933,12.825817,3.077672,2.387202,1.195597,18.570237,0.074675,2.482539,8.270859,1.889832
std,0.108413,1.289836,6.215327,2.210685,1.895446,1.136992,5.388191,0.098156,2.832190,6.093909,1.737911
min,0.000000,-2.000000,-54.400000,0.000000,0.000000,-0.600000,0.000000,-1.312000,-2.100000,0.000000,0.000000
25%,0.000000,-0.100000,10.000000,1.400000,1.000000,0.300000,15.000000,0.040000,0.300000,3.700000,0.700000
50%,0.000000,0.100000,12.800000,2.500000,1.900000,0.900000,18.100000,0.082000,1.600000,6.700000,1.400000
75%,0.000000,0.900000,15.900000,4.300000,3.300000,1.800000,21.700000,0.120000,3.800000,11.500000,2.500000
max,1.000000,11.800000,133.800000,12.200000,11.600000,9.100000,72.500000,2.712000,20.300000,36.100000,13.100000


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12697 entries, 0 to 12696
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   points_share  12697 non-null  float64
 1   VORP          12697 non-null  float64
 2   PER           12697 non-null  float64
 3   FG            12697 non-null  float64
 4   2P            12697 non-null  float64
 5   DWS           12697 non-null  float64
 6   USG%          12697 non-null  float64
 7   WS/48         12697 non-null  float64
 8   WS            12697 non-null  float64
 9   PTS           12697 non-null  float64
 10  FTA           12697 non-null  float64
dtypes: float64(11)
memory usage: 1.1 MB


---
#### Select target variable and features

* The target variable chosen is points_share, which represents a player's share of All-NBA vote points.
* Predictor variables include stats selected by random forest feature importance, excluding direct vote metrics.

In [11]:
# Assign target variable
y = data["points_share"]

In [21]:
# Assign features
X = data[['VORP', 'PER', 'FG', '2P', 'DWS', 'USG%', 'WS/48', 'WS', 'PTS', 'FTA']]

---
#### Scale data for modeling

* Features were standardized using StandardScaler to ensure fair weight comparisons across metrics with different units.
* Target variable was left unscaled to retain interpretability of points_share.

In [13]:
# Import StandardScaler
from sklearn.preprocessing import StandardScaler

In [14]:
# For newer versions of Python/Jupyter, change default so that method config attributes always show
from sklearn import set_config
set_config(print_changed_only=False, display="diagram")

In [23]:
# Scale data
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_scaled.head()

,VORP,PER,FG,2P,DWS,USG%,WS/48,WS,PTS,FTA
0,0.866863,0.494633,1.231489,0.323313,0.531601,0.580879,0.767434,1.453865,1.448904,0.696362
1,0.556734,1.299127,-0.035136,0.270553,0.443646,0.580879,0.950823,0.359263,-0.241375,-0.454490
2,-0.451186,-1.597053,-1.256525,-1.101210,-1.051586,0.265362,-1.606439,-0.876578,-1.143951,-0.684661
3,3.813089,1.862273,1.638619,1.484035,1.499104,1.044875,1.388919,2.901565,1.678651,1.329331
4,2.805169,1.250858,1.955275,1.114715,0.707511,1.638790,0.767434,2.160060,2.269428,1.962300


---
#### Run VIF iteratively to remove multicolinearity

* Variance Inflation Factor (VIF) was calculated to identify multicollinearity between predictors.
* Features with VIF > 10 were removed in successive iterations, resulting in a set of statistically independent features.

In [25]:
# Import VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [27]:
# Remove VIF Iteratively

def calculate_vif(df, thresh=10.0):
    variables = df.columns.tolist()
    dropped = True
    while dropped:
        dropped = False
        vif_data = pd.DataFrame()
        vif_data["Feature"] = variables
        vif_data["VIF"] = [variance_inflation_factor(df[variables].values, i) for i in range(len(variables))]

        max_vif = vif_data["VIF"].max()
        if max_vif > thresh:
            drop_feature = vif_data.sort_values("VIF", ascending=False)["Feature"].iloc[0]
            print(f"Dropping '{drop_feature}' with VIF: {max_vif:.2f}")
            variables.remove(drop_feature)
            dropped = True

# Usage
cleaned_df = calculate_vif(X_scaled, thresh=10)

Dropping 'FG' with VIF: 887.84
Dropping 'PER' with VIF: 18.45
Dropping 'WS' with VIF: 12.32


In [31]:
# Show final columns for modeling along with VIF
X_scaled_final = X_scaled[["USG%", "DWS", "VORP", "WS/48", "FTA", "2P", "PTS"]]

vif_final = pd.DataFrame()
vif_final["Features"] = X_scaled_final.columns
vif_final["VIF"] = [variance_inflation_factor(X_scaled_final.values, i) for i in range(X_scaled_final.shape[1])]

vif_final.sort_values(by="VIF", ascending=True)

,Features,VIF
3,WS/48,1.385923
0,USG%,1.979146
1,DWS,2.578201
2,VORP,3.122776
4,FTA,5.067715
5,2P,7.020389
6,PTS,8.144588


---
#### OLS Regression

* A linear regression model was trained and evaluated 10 times using randomly seeded train-test splits to assess stability and performance.
* R-squared scores across the runs ranged from 0.3816 to 0.4783, indicating that the model explained roughly 38% to 48% of the variance in points_share.
* MAE (Mean Absolute Error) remained consistently low between 0.0391 and 0.0416, suggesting that predictions were close to actual values on average.
* RMSE (Root Mean Squared Error) ranged from 0.0767 to 0.0855, supporting a relatively small typical prediction error despite the modest R-squared.
* While error metrics indicate the model performs reasonably well, the low R-squared implies that linear regression may not capture all the complexity in what determines All-NBA vote share.

P-values and weights
* The feature VORP has the highest positive average weight (0.0800), suggesting strong positive association with points_share.
* FTA (free throw attempts) is also positively associated, with an average weight of 0.0315.
* USG% has a small positive average weight, but an extremely small p-value (9.04e-237), indicating high statistical significance.
* Features like WS/48 and DWS have negative weights, suggesting inverse relationships with points_share.
* Despite negative weights, all listed features have very small p-values, supporting their inclusion in the regression model as statistically significant.

In [40]:
# Import Linear Regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import f_regression
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

In [68]:
# Adjusted R-Squared function
def adjusted_r2(r2, n, p):
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

In [89]:
# Run OLS 10 times with random seed value each time
ols_results = []

# Create dataframe for p-values and weights
ols_p_values_matrix = pd.DataFrame(index=X_scaled_final.columns)
ols_weights_matrix = pd.DataFrame(index=X_scaled_final.columns)

# Set number of model runs in loop
num_runs = 10

for i in range(num_runs):
    seed = np.random.randint(1, 100)

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X_scaled_final, y, test_size=0.2, random_state=seed)

    # Linear regression
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predict results
    y_pred = model.predict(X_test)

    # Compute Adjusted R-squared
    r2 = r2_score(y_test, y_pred)
    n = X_test.shape[0]
    p = X_test.shape[1]
    adj_r2 = adjusted_r2(r2, n, p)
    
    # Log results
    ols_results.append({
        "Run_Number" : i + 1,
        "Random_State" : seed,
        "Adj_R_Squared" : round(adj_r2, 4),
        "MAE": round(mean_absolute_error(y_test, y_pred), 4),
        "RMSE": round(root_mean_squared_error(y_test, y_pred), 4)
    })
    p_values = f_regression(X_train, y_train)[1]
    weights = model.coef_

    # Store results
    ols_p_values_matrix[f'run_{i+1}'] = pd.Series(p_values, index=X_train.columns)
    ols_weights_matrix[f'run_{i+1}'] = pd.Series(weights, index=X_train.columns)

# Convert results to dataframe
ols_results_df = pd.DataFrame(ols_results)
ols_results_df

,Run_Number,Random_State,Adj_R_Squared,MAE,RMSE
0,1,95,0.4225,0.0402,0.0800
1,2,43,0.3695,0.0396,0.0790
2,3,67,0.4172,0.0424,0.0866
3,4,89,0.4756,0.0418,0.0892
4,5,14,0.4070,0.0406,0.0806
5,6,32,0.4041,0.0401,0.0756
6,7,44,0.4418,0.0417,0.0851
7,8,45,0.4412,0.0417,0.0834
8,9,71,0.3245,0.0383,0.0716
9,10,3,0.4238,0.0406,0.0788


In [77]:
# P-values and weights
ols_combined_summary = pd.DataFrame({
    'Avg_Weight': weights_matrix.mean(axis=1).round(4),
    'Avg_p_value': p_values_matrix.mean(axis=1)
}).sort_values(by='Avg_Weight', ascending=False)

ols_combined_summary

,Avg_Weight,Avg_p_value
VORP,0.0798,0.000000e+00
FTA,0.0310,0.000000e+00
USG%,0.0076,1.367254e-226
2P,0.0060,0.000000e+00
WS/48,-0.0047,1.555128e-103
DWS,-0.0136,0.000000e+00
PTS,-0.0388,0.000000e+00


---
#### Elastic Net Regression

In [95]:
# Import library
from sklearn.linear_model import ElasticNet

In [96]:
# Run Elastic net 10 times with random seed value each time
enet_results = []

# Set number of model runs in loop
num_runs = 10

for i in range(num_runs):
    seed = np.random.randint(1, 100)

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X_scaled_final, y, test_size=0.2, random_state=seed)

    # Elastic net
    model = ElasticNet(alpha=0.005, l1_ratio=0.5, max_iter=10000, random_state=seed)
    model.fit(X_train, y_train)

    # Predict results
    y_pred = model.predict(X_test)

    # Compute Adjusted R-squared
    r2 = r2_score(y_test, y_pred)
    n = X_test.shape[0]
    p = X_test.shape[1]
    adj_r2 = adjusted_r2(r2, n, p)
    
    # Log results
    enet_results.append({
        "Run_Number" : i + 1,
        "Random_State" : seed,
        "Adj_R_Squared" : round(adj_r2, 4),
        "MAE": round(mean_absolute_error(y_test, y_pred), 4),
        "RMSE": round(root_mean_squared_error(y_test, y_pred), 4)
    })

# Convert results to dataframe
enet_results_df = pd.DataFrame(enet_results)
enet_results_df

,Run_Number,Random_State,Adj_R_Squared,MAE,RMSE
0,1,18,0.3175,0.0391,0.0716
1,2,72,0.3739,0.0378,0.0771
2,3,34,0.3615,0.0398,0.0795
3,4,13,0.4027,0.0396,0.0800
4,5,39,0.3961,0.0400,0.0844
5,6,15,0.4191,0.0392,0.0823
6,7,69,0.4151,0.0397,0.0788
7,8,85,0.3866,0.0378,0.0781
8,9,83,0.4218,0.0398,0.0868
9,10,64,0.4164,0.0375,0.0796


---
#### XGBoost

In [100]:
# Import library
from xgboost import XGBRegressor

In [102]:
# Run XGBoost 10 times with random seed value each time
xgb_results = []

# Set number of model runs in loop
num_runs = 10

for i in range(num_runs):
    seed = np.random.randint(1, 100)

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X_scaled_final, y, test_size=0.2, random_state=seed)

    # XGBoost
    model = XGBRegressor(learning_rate=0.05, n_estimators=500, max_depth=4, random_state=seed)
    model.fit(X_train, y_train)

    # Predict results
    y_pred = model.predict(X_test)

    # Compute Adjusted R-squared
    r2 = r2_score(y_test, y_pred)
    n = X_test.shape[0]
    p = X_test.shape[1]
    adj_r2 = adjusted_r2(r2, n, p)
    
    # Log results
    xgb_results.append({
        "Run_Number" : i + 1,
        "Random_State" : seed,
        "Adj_R_Squared" : round(adj_r2, 4),
        "MAE": round(mean_absolute_error(y_test, y_pred), 4),
        "RMSE": round(root_mean_squared_error(y_test, y_pred), 4)
    })

# Convert results to dataframe
xgb_results_df = pd.DataFrame(xgb_results)
xgb_results_df

,Run_Number,Random_State,Adj_R_Squared,MAE,RMSE
0,1,38,0.7329,0.0102,0.0573
1,2,49,0.7876,0.0103,0.0530
2,3,90,0.7082,0.0107,0.0576
3,4,11,0.7302,0.0100,0.0534
4,5,96,0.8009,0.0091,0.0474
5,6,5,0.7701,0.0101,0.0545
6,7,84,0.7297,0.0112,0.0581
7,8,30,0.7723,0.0097,0.0533
8,9,15,0.7246,0.0108,0.0567
9,10,53,0.7876,0.0113,0.0598
